# Research Paper Ingestion & Processing

In [14]:
%pip install arxiv langchain langchain_community pypdf

Note: you may need to restart the kernel to use updated packages.


In [15]:
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tempfile
import os
import time

## Fetching Research Papers from ArXiv
- Query ArXiv for academic papers based on a search term.
- Instead of storing PDFs, we load them **temporarily** then extract the text.

In [16]:
def fetch_papers(query="LLM", max_results=5):
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results, sort_order=arxiv.SortOrder.Descending)
    
    all_chunks = []  
    
    for result in client.results(search):
        print(f"Processing: {result.title}")
        
        with tempfile.NamedTemporaryFile(delete=True, suffix=".pdf") as temp_pdf:
            try:
                result.download_pdf(filename=temp_pdf.name)
                print(f"-> Downloaded: {result.title}")
                
                loader = PyPDFLoader(temp_pdf.name)
                papers = loader.load()
                
                full_text = " ".join(p.page_content for p in papers)
                
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                paper_chunks = text_splitter.create_documents([full_text])
                
                all_chunks.extend(paper_chunks) 
                
            except (FileNotFoundError, ConnectionResetError) as e:
                print("Error occurred while downloading or processing PDF:", e)
                time.sleep(5)
    
    print(f"Total chunks extracted: {len(all_chunks)}")
    return all_chunks

## Running the Paper Retrieval Pipeline
Testing the function by retrieving **3 recent papers** related to **Large Language Models**.

In [17]:
query = "Large Language Models"
paper_chunks = fetch_papers(query=query, max_results=3)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")

Processing: Lost in Translation: Large Language Models in Non-English Content Analysis
-> Downloaded: Lost in Translation: Large Language Models in Non-English Content Analysis
Processing: Cedille: A large autoregressive French language model
-> Downloaded: Cedille: A large autoregressive French language model
Processing: How Good are Commercial Large Language Models on African Languages?
-> Downloaded: How Good are Commercial Large Language Models on African Languages?
Total chunks extracted: 480

Chunk 1:
Lost in Translation
May 2023
A report from
Gabriel Nicholas 
Aliya Bhatia
Large Language Models in 
Non-English Content Analysis GABRIEL NICHOLAS
Research Fellow at the Center for Democracy & T echnology.
ALIYA BHATIA
Policy Analyst, Free Expression Project at the Center for 
Democracy & T echnology...

Chunk 2:
civil liberties in the digital age. W e shape technology policy, governance, 
and design with a focus on equity and democratic values. Established in 
1996, CDT has been a t